# Player Analysis: Einzelne Spieler

Analyse von individuellen Spielern: Anzahl Spieler, Elo-Entwicklung, Top-Spieler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Lade Daten
df = pd.read_csv('games.csv')
df['created_at'] = pd.to_datetime(df['created_at'], unit='ms')
df = df.sort_values('created_at')

# Erstelle Player-Listen
white_players = df[['white_id', 'white_rating']].copy()
white_players.columns = ['player_id', 'elo']
white_players['color'] = 'White'

black_players = df[['black_id', 'black_rating']].copy()
black_players.columns = ['player_id', 'elo']
black_players['color'] = 'Black'

# Kombiniere alle Spieler
all_players = pd.concat([white_players, black_players], ignore_index=True)
all_players['created_at'] = pd.concat([df['created_at'], df['created_at']], ignore_index=True).values

# Zähle eindeutige Spieler
unique_white = df['white_id'].nunique()
unique_black = df['black_id'].nunique()
unique_total = len(set(df['white_id'].unique()) | set(df['black_id'].unique()))

print(f"Spieler-Statistiken:")
print(f"  Gesamte Spiele: {len(df):,}")
print(f"  Eindeutige White-Spieler: {unique_white:,}")
print(f"  Eindeutige Black-Spieler: {unique_black:,}")
print(f"  Eindeutige Spieler (gesamt): {unique_total:,}")

In [ ]:
# Top Spieler nach Anzahl Spiele
white_games = df.groupby('white_id').size().reset_index(name='games_white')
black_games = df.groupby('black_id').size().reset_index(name='games_black')

# Merge
player_stats = white_games.copy()
player_stats.columns = ['player_id', 'games_white']

black_games.columns = ['player_id', 'games_black']
player_stats = player_stats.merge(black_games, on='player_id', how='outer').fillna(0)

player_stats['total_games'] = player_stats['games_white'] + player_stats['games_black']
player_stats = player_stats.sort_values('total_games', ascending=False)

print("\nTop 10 Spieler nach Anzahl Spiele:")
print(player_stats.head(10))

# Höchste und niedrigste Activity
most_active_player = player_stats.iloc[0]
most_active_id = most_active_player['player_id']
most_active_games = most_active_player['total_games']

print(f"\nMeist aktiver Spieler: {most_active_id}")
print(f"  Spiele: {most_active_games:.0f} (White: {most_active_player['games_white']:.0f}, Black: {most_active_player['games_black']:.0f})")

In [ ]:
# Verteilung der Spielanzahl
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Spieler-Aktivität Verteilung', fontsize=14, fontweight='bold')

# Histogram: Spiele pro Spieler
axes[0].hist(player_stats['total_games'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(player_stats['total_games'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {player_stats["total_games"].mean():.0f}')
axes[0].axvline(player_stats['total_games'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {player_stats["total_games"].median():.0f}')
axes[0].set_xlabel('Anzahl Spiele pro Spieler')
axes[0].set_ylabel('Anzahl Spieler')
axes[0].set_title('Verteilung der Spielanzahl')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box Plot
axes[1].boxplot([player_stats['total_games']], labels=['Total Games'], patch_artist=True)
axes[1].set_ylabel('Anzahl Spiele')
axes[1].set_title('Box Plot: Spiele pro Spieler')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Statistiken
print(f"\nSpiele pro Spieler - Statistiken:")
print(f"  Mean: {player_stats['total_games'].mean():.1f}")
print(f"  Median: {player_stats['total_games'].median():.1f}")
print(f"  Std Dev: {player_stats['total_games'].std():.1f}")
print(f"  Min: {player_stats['total_games'].min():.0f}")
print(f"  Max: {player_stats['total_games'].max():.0f}")
print(f"  Q1: {player_stats['total_games'].quantile(0.25):.0f}")
print(f"  Q3: {player_stats['total_games'].quantile(0.75):.0f}")

In [ ]:
# Analysiere Elo-Entwicklung des aktivsten Spielers
player_games = pd.concat([
    df[df['white_id'] == most_active_id][['created_at', 'white_rating']].rename(columns={'white_rating': 'elo'}),
    df[df['black_id'] == most_active_id][['created_at', 'black_rating']].rename(columns={'black_rating': 'elo'})
])

player_games = player_games.sort_values('created_at').reset_index(drop=True)

print(f"\nElo-Entwicklung: Spieler {most_active_id}")
print(f"  Anfangs-Elo: {player_games['elo'].iloc[0]:.0f}")
print(f"  End-Elo: {player_games['elo'].iloc[-1]:.0f}")
print(f"  Veränderung: {player_games['elo'].iloc[-1] - player_games['elo'].iloc[0]:+.0f}")
print(f"  Min Elo: {player_games['elo'].min():.0f}")
print(f"  Max Elo: {player_games['elo'].max():.0f}")
print(f"  Durchschn. Elo: {player_games['elo'].mean():.0f}")
print(f"  Std Dev: {player_games['elo'].std():.0f}")

In [ ]:
# Visualisiere Elo-Entwicklung des aktivsten Spielers
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle(f'Elo-Entwicklung: Spieler {most_active_id} ({len(player_games)} Spiele)', fontsize=14, fontweight='bold')

# 1. Elo über Zeit
axes[0].plot(player_games.index, player_games['elo'], marker='o', linestyle='-', linewidth=1, markersize=3, alpha=0.7, color='steelblue')
axes[0].axhline(player_games['elo'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {player_games["elo"].mean():.0f}')
axes[0].fill_between(player_games.index, player_games['elo'], alpha=0.2, color='steelblue')
axes[0].set_ylabel('Elo Rating')
axes[0].set_title('Elo-Verlauf über Spiele')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Moving Average (100 Spiele)
player_games['elo_ma30'] = player_games['elo'].rolling(window=30, center=True).mean()
player_games['elo_ma100'] = player_games['elo'].rolling(window=100, center=True).mean()

axes[1].plot(player_games.index, player_games['elo'], marker='.', linestyle='-', linewidth=0.5, alpha=0.3, color='lightgray', label='Täglich')
axes[1].plot(player_games.index, player_games['elo_ma30'], linewidth=2, color='orange', label='30-Spiele MA')
axes[1].plot(player_games.index, player_games['elo_ma100'], linewidth=2, color='red', label='100-Spiele MA')
axes[1].set_ylabel('Elo Rating')
axes[1].set_xlabel('Spiel-Nummer')
axes[1].set_title('Elo mit Moving Averages')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analysiere Elo-Verteilung des aktivsten Spielers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Elo-Statistiken: Spieler {most_active_id}', fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(player_games['elo'], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(player_games['elo'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {player_games["elo"].mean():.0f}')
axes[0].axvline(player_games['elo'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {player_games["elo"].median():.0f}')
axes[0].set_xlabel('Elo Rating')
axes[0].set_ylabel('Häufigkeit')
axes[0].set_title('Elo-Verteilung')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box Plot mit Violin Plot
parts = axes[1].violinplot([player_games['elo']], positions=[1], showmeans=True, showmedians=True)
axes[1].set_xticks([1])
axes[1].set_xticklabels(['Elo'])
axes[1].set_ylabel('Elo Rating')
axes[1].set_title('Elo-Verteilung (Violin Plot)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 Spieler Vergleich
top_20_players = player_stats.head(20).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 20 Spieler nach Spielanzahl', fontsize=14, fontweight='bold')

# Bar Chart: Spiele
colors = ['green' if pid == most_active_id else 'steelblue' for pid in top_20_players['player_id']]
axes[0].barh(range(len(top_20_players)), top_20_players['total_games'], color=colors, alpha=0.7, edgecolor='black')
axes[0].set_yticks(range(len(top_20_players)))
axes[0].set_yticklabels(top_20_players['player_id'])
axes[0].set_xlabel('Gesamte Spiele')
axes[0].set_title('Anzahl Spiele')
axes[0].invert_yaxis()
axes[0].grid(alpha=0.3, axis='x')

# White vs Black
x = np.arange(len(top_20_players))
width = 0.35
axes[1].bar(x - width/2, top_20_players['games_white'], width, label='White', alpha=0.7, color='blue')
axes[1].bar(x + width/2, top_20_players['games_black'], width, label='Black', alpha=0.7, color='red')
axes[1].set_ylabel('Anzahl Spiele')
axes[1].set_title('White vs Black Spielanzahl')
axes[1].set_xticks(x)
axes[1].set_xticklabels(top_20_players['player_id'], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Zusammenfassung
print("\n" + "="*70)
print("ZUSAMMENFASSUNG: PLAYER ANALYSIS")
print("="*70)

print(f"\n👥 SPIELER")
print(f"   • Gesamte Spiele: {len(df):,}")
print(f"   • Eindeutige Spieler: {unique_total:,}")
print(f"   • White-Spieler: {unique_white:,}")
print(f"   • Black-Spieler: {unique_black:,}")

print(f"\n📊 AKTIVITÄT")
print(f"   • Mean Spiele/Spieler: {player_stats['total_games'].mean():.1f}")
print(f"   • Median Spiele/Spieler: {player_stats['total_games'].median():.1f}")
print(f"   • Min: {player_stats['total_games'].min():.0f}")
print(f"   • Max: {player_stats['total_games'].max():.0f}")

print(f"\n🏆 MEISTAKTIVER SPIELER")
print(f"   • ID: {most_active_id}")
print(f"   • Gesamte Spiele: {most_active_games:.0f}")
print(f"   • White: {most_active_player['games_white']:.0f}, Black: {most_active_player['games_black']:.0f}")

print(f"\n📈 ELO-ENTWICKLUNG ({most_active_id})")
print(f"   • Start-Elo: {player_games['elo'].iloc[0]:.0f}")
print(f"   • End-Elo: {player_games['elo'].iloc[-1]:.0f}")
print(f"   • Veränderung: {player_games['elo'].iloc[-1] - player_games['elo'].iloc[0]:+.0f}")
print(f"   • Range: {player_games['elo'].min():.0f} - {player_games['elo'].max():.0f}")
print(f"   • Durchschnitt: {player_games['elo'].mean():.0f}")